# Acoustic Insect Species Identification — CNN & EfficientNetV2 Training
### ICT-4442 Deep Learning Mini Project | Author: Saurabh Tiwari (230911238)

This notebook covers:
1. **Dataset loading & 60/20/20 stratified splitting**
2. **VGG-Style Log-Mel CNN training**
3. **EfficientNetV2-S Transfer Learning fine-tuning**
4. **Auxiliary Temperature Conditioning Ablation Study** (Dolbear's Law)
5. **Comparative Metrics & Long-Tail Tier Analysis** (Head, Medium, Tail breakdown)
6. **Confusion Matrix visualization**

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.dataset import SyntheticInsectDataset, create_dataloaders
from src.models.vgg_cnn import LogMelCNN
from src.models.efficientnet import InsectEfficientNetV2
from src.utils.trainer import ModelTrainer
from src.utils.metrics import calculate_metrics, analyze_long_tail_performance, plot_training_curves, plot_confusion_matrix_heatmap

## 1. Load Dataset & Create Stratified Splits

In [ ]:
dataset = SyntheticInsectDataset(num_samples=1200, num_classes=20, chunk_duration=5.0, long_tail=True, random_seed=42)
train_loader, val_loader, test_loader, split_info = create_dataloaders(dataset, batch_size=32, in_channels=1)

print("Split summary:", split_info)

## 2. Train VGG-Style Log-Mel CNN Baseline

In [ ]:
vgg_model = LogMelCNN(num_classes=20, in_channels=1, base_channels=32, use_temperature=True, dropout_rate=0.3)
optimizer = torch.optim.AdamW(vgg_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8, eta_min=1e-5)

trainer_vgg = ModelTrainer(
    model=vgg_model,
    optimizer=optimizer,
    scheduler=scheduler,
    checkpoint_dir='../checkpoints',
    model_name='vgg_cnn_notebook',
    use_temperature=True
)

history_vgg = trainer_vgg.fit(train_loader, val_loader, epochs=6, early_stopping_patience=4)

## 3. Evaluate VGG CNN on Test Set & Long-Tail Tiers

In [ ]:
test_loss, test_metrics, y_true, y_pred = trainer_vgg.evaluate(test_loader)
print(f"VGG-CNN Test Accuracy: {test_metrics['accuracy']*100:.2f}%")
print(f"VGG-CNN Test Macro-F1: {test_metrics['macro_f1']*100:.2f}%")

class_counts = pd.Series(y_true).value_counts().to_dict()
tier_results = analyze_long_tail_performance(y_true, y_pred, class_counts, head_threshold=15, tail_threshold=5)
print("\nLong-Tail Breakdown:", tier_results)

## 4. Visualizations: Loss Curves & Confusion Matrix

In [ ]:
plot_training_curves(history_vgg)
plot_confusion_matrix_heatmap(y_true, y_pred, max_classes_to_show=10)
plt.show()